# Phase 1: Data Collection and Preparation

**DS588 Algorithmic Portfolio Management**

This notebook turns the raw downloads into clean, documented datasets for the rest of the project.

**Principle:** Phase 1 does not choose investments. It only removes assets that cannot be used, and every removal has a documented reason (access rules, the project's no-short / no-borrowing rule, missing or broken data, or no actual trading). Choosing among the remaining assets is the job of the Phase 2 algorithm.

| Step | What happens |
|---|---|
| 0 | Settings: folders, study window, filter rules (all in one place) |
| 1 | Load Turkish funds (TEFAS monthly exports) |
| 2 | Load US ETFs (`abd_etf_download.py` output) |
| 3 | Load the USD/TRY exchange rate |
| 4 | Data quality checks (coverage, calendars, suspicious jumps) |
| 5 | Eligibility filters (remove assets that cannot be used, with the reason) |
| 6 | Eligible universe (every asset that passed; no hand-picking) |
| 7 | Save outputs |
| 8 | Final validation checks |

**Other files:** `abd_etf_download.py` (US download script), `DATA_DICTIONARY.md` (every column of every output), `README.md` (how to reproduce everything), `requirements.txt` (Python packages).

**How to run:** open this notebook from the project root (`DS588_project/`) and run all cells top to bottom.
Nothing is downloaded except the exchange rate (step 3), and only if it is not already saved.

**Expected folder layout**
```
DS588_project/
├── Data/
│   ├── Raw/
│   │   ├── tr/   Fon_Verileri_CSV_Tum_Veri_<export-date>_<month>_<borsa|menkul>.csv
│   │   ├── us/   us_etf_daily_2026-04-01_2026-10-01.csv
│   │   └── fx/   (created by this notebook)
│   └── Processed/ (created by this notebook)
└── phase1_preparation.ipynb
```

## Raw data: where it comes from and how to reproduce it

**US ETFs (automated).** `abd_etf_download.py` reads the official Nasdaq Trader symbol lists (`nasdaqlisted.txt`, `otherlisted.txt`), keeps the rows marked as ETF (and not test issues), and downloads daily prices from Yahoo Finance with `yfinance` in batches of 20 tickers with a 3-second pause. It can be stopped and restarted; it continues with the missing tickers. Output: `Data/Raw/us/us_etf_daily_2026-04-01_2026-10-01.csv`, with a `retrieved_at_utc` timestamp on every row.

**Turkish funds (manual export, data access limitation).** I first tried to download TEFAS data programmatically, but repeated requests timed out and each request returned at most about one month of data, so collecting six months for every fund this way was not practical. The reproducible alternative I used is the manual export on the TEFAS website:

1. Open the historical data page of [tefas.gov.tr](https://www.tefas.gov.tr).
2. Choose the fund type: **Borsa Yatırım Fonları** (exchange-traded funds) or **Menkul Kıymet Yatırım Fonları** (securities investment funds).
3. Set the date range to one calendar month (April, May, June, July, August; for September use 01.09.2026 to 01.10.2026 so the window end is included).
4. Select all funds ("Tümünü seç") and export the full data as CSV.
5. Save the file in `Data/Raw/tr/` and keep `borsa` or `menkul` in the file name; the notebook uses this word to tell the fund type.

This gives 12 files (6 months x 2 fund types). My exports were made on 9 and 10 October 2026 (the date is in each file name).

**Not used: pension funds (Emeklilik Yatırım Fonları, BES).** They can only be held inside a private pension contract and cannot be bought or sold freely like ordinary investment funds, so they are outside the project's investment scope and were not exported.

**USD/TRY.** Downloaded once by this notebook (step 3) from Yahoo Finance (`USDTRY=X`) and saved.

**Redistribution.** The raw files come from TEFAS and Yahoo Finance. Where redistribution is not allowed, the steps above (and `abd_etf_indir.py`) are my retrieval instructions.

## 0. Settings

Every decision that affects the data is a named setting here, so it is easy to explain in the report and easy to change.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---------- Folders (paths are relative to the project root) ----------
RAW_TR = Path("Data/Raw/tr")
RAW_US = Path("Data/Raw/us")
RAW_FX = Path("Data/Raw/fx")
OUT = Path("Data/Processed")

# ---------- Study window (both dates included) ----------
START = pd.Timestamp("2026-04-01")
END = pd.Timestamp("2026-10-01")

US_FILE = RAW_US / f"us_etf_daily_{START.date()}_{END.date()}.csv"
FX_FILE = RAW_FX / f"usdtry_daily_{START.date()}_{END.date()}.csv"
FX_TICKER = "USDTRY=X"  # Yahoo Finance: how many TRY one USD costs

# ---------- Eligibility rules ----------
# Phase 1 does NOT pick investments. Each rule below removes assets that cannot be used,
# for the reason written next to it. There is no size, return or popularity filter.
MIN_COVERAGE = 0.98          # data: asset needs a price on >= 98% of its market's trading days
MAX_DAILY_MOVE_TR = 0.25     # data: a TR fund moving more than 25% in one day is treated as a data error
MAX_DAILY_MOVE_US = 0.40     # data: same idea for US ETFs (leveraged products are removed separately)
US_TRADED_DAYS_WINDOW = 20   # tradability: look at the FIRST 20 trading days only (no look-ahead)
MIN_US_TRADED_SHARE = 1.0    # tradability: ETF must have volume > 0 on all of those days,
                             # otherwise its "close" is not a price anyone actually traded at

# Project rule: "nonnegative weights with no short selling or borrowing".
# A leveraged ETF (2x, 3x, "Ultra") is the same as buying the index with borrowed money, and an
# inverse ETF ("Short", "Bear", -1x) is the same as shorting it, so holding one breaks that rule.
# The Nasdaq lists have no leverage flag, so they are detected from the fund name.
# NOT matched (normal long funds, checked against the real ETF list): ultra-short / short-term /
# short muni bond funds, "Ultra Buffer", "Ultra Dividend", "Ultra-Small", "Daily Income" and
# actively managed long/short equity funds (their aim is not a multiple or the opposite of an index).
US_EXCLUDE_PATTERN = (
    r"\b\d(?:\.\d+)?X\b|-\dX\b|\bULTRAPRO\b|\bULTRASHORT\b|\bULTRA\b(?![- ]SHORT|-SMALL| DIVIDEND| BUFFER| OPTION)|"
    r"LEVERAGED|\bLEVERAGE\b|INVERSE|\bBEAR\b|\bBULL\b|"
    r"(?<!ULTRA-)(?<!ULTRA )(?<!LONG/)(?<!LONG-)(?<!LONG )\bSHORT\b"
    r"(?![- ](?:TERM|DURATION|MATURITY|TREASURY|INCOME|BOND|MUNI|MUNICIPAL|HIGH YIELD MUNI|HORIZON))"
)

OUT.mkdir(parents=True, exist_ok=True)
RAW_FX.mkdir(parents=True, exist_ok=True)
print(f"Window: {START.date()} to {END.date()}")

Window: 2026-04-01 to 2026-10-01


## 1. Turkish funds (TEFAS)

TEFAS exports start with a few info lines (export date, record count) before the real table, so the reader looks for the header line that starts with `Fon Kodu,`.

**Two kinds of Turkish funds** (the project asks to keep them apart):
- **BYF** (*borsa yatırım fonu*, exchange-traded fund): trades on Borsa İstanbul like a stock. I use the **exchange bulletin price** (`Borsa Bülten Fiyatı`), because that is the price you can actually trade at.
- **YAT** (*menkul kıymet yatırım fonu*, mutual fund): bought and sold through TEFAS at the fund's **unit price** (`Fiyat`, the net asset value per unit).

**Number format:** TEFAS writes decimals with a comma (`45,34301`), so commas are converted to dots.
**Date format:** TEFAS dates are `DD.MM.YYYY`; I parse them with that exact format, so a date that does not match stops the notebook instead of being read wrongly (a mixed format can silently swap day and month).

In [15]:
def read_tefas_csv(path):
    """Read one TEFAS export, skipping the info lines above the table."""
    with open(path, encoding="utf-8-sig") as f:
        for header_row, line in enumerate(f):
            if line.startswith("Fon Kodu,"):
                break
        else:
            raise ValueError(f"No 'Fon Kodu' header found in {path.name}")
    return pd.read_csv(path, skiprows=header_row, dtype=str)


def tr_decimal(series):
    """'45,34301' -> 45.34301 (TEFAS uses a decimal comma, no thousands separator)."""
    return pd.to_numeric(series.str.replace(",", ".", regex=False), errors="coerce")


frames = []
for path in sorted(RAW_TR.glob("*.csv")):
    name = path.stem.lower()
    if "borsa" in name:
        fund_type = "BYF"
    elif "menkul" in name:
        fund_type = "YAT"
    else:
        print(f"Skipped (cannot tell fund type from file name): {path.name}")
        continue
    df = read_tefas_csv(path)
    df["fund_type"] = fund_type
    df["source_file"] = path.name
    frames.append(df)

tr_raw = pd.concat(frames, ignore_index=True)
tr_raw = tr_raw.rename(columns={
    "Fon Kodu": "symbol", "Fon Adı": "name", "Tarih": "date_text", "Fiyat": "nav",
    "Fon Toplam Değer": "fund_size_tl", "Borsa Bülten Fiyatı": "bulletin_price",
    "Kişi Sayısı": "investors", "Tedavüldeki Pay Sayısı": "shares_outstanding",
})
tr_raw["date"] = pd.to_datetime(tr_raw["date_text"], format="%d.%m.%Y")  # strict: fails loudly if a date looks different
tr_raw["nav"] = tr_decimal(tr_raw["nav"])
tr_raw["fund_size_tl"] = tr_decimal(tr_raw["fund_size_tl"])
tr_raw["shares_outstanding"] = pd.to_numeric(tr_raw["shares_outstanding"], errors="coerce")  # already dot-decimal
tr_raw["investors"] = pd.to_numeric(tr_raw["investors"], errors="coerce")
tr_raw["bulletin_price"] = tr_decimal(tr_raw["bulletin_price"]) if "bulletin_price" in tr_raw else np.nan

# Price used in the project: exchange price for BYFs, unit price for mutual funds
tr_raw["price"] = np.where(tr_raw["fund_type"] == "BYF", tr_raw["bulletin_price"], tr_raw["nav"])
tr_raw["price_type"] = np.where(tr_raw["fund_type"] == "BYF", "bist_bulletin_close", "tefas_unit_price")

print(f"Files loaded: {len(frames)} | rows: {len(tr_raw):,} | funds: {tr_raw['symbol'].nunique():,}")

Files loaded: 12 | rows: 255,138 | funds: 2,091


### 1.1 Coverage per file

Each monthly export should contain roughly the same funds. If one month has far fewer funds than the others, a filter was probably left on in TEFAS while exporting, and that month should be downloaded again.

In [16]:
coverage_by_file = (tr_raw.groupby(["fund_type", "source_file"])
                    .agg(first_date=("date", "min"), last_date=("date", "max"),
                         trading_days=("date", "nunique"), funds=("symbol", "nunique"), rows=("symbol", "size"))
                    .reset_index().sort_values(["fund_type", "first_date"]))
typical = coverage_by_file.groupby("fund_type")["funds"].transform("max")
coverage_by_file["looks_incomplete"] = coverage_by_file["funds"] < 0.5 * typical
display(coverage_by_file)

bad = coverage_by_file[coverage_by_file["looks_incomplete"]]
if len(bad):
    print("WARNING: these exports have less than half the usual number of funds.")
    print("Funds missing from these months cannot pass the coverage filter in step 5.")
    print("Re-export them from TEFAS with no fund-type filter:")
    for f in bad["source_file"]:
        print("   -", f)

,fund_type,source_file,first_date,last_date,trading_days,funds,rows,looks_incomplete
4,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-10_nisan_bor...,2026-04-01,2026-04-30,21,30,630,False
3,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-10_mayıs_bor...,2026-05-04,2026-05-26,16,30,480,False
2,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-10_haziran_b...,2026-06-01,2026-06-30,22,30,660,False
5,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-10_temmuz_bo...,2026-07-01,2026-07-31,22,30,660,False
1,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-10_agustos_b...,2026-08-03,2026-08-31,21,30,630,False
0,BYF,Fon_Verileri_CSV_Tum_Veri_2026-10-09_eylul_bor...,2026-09-01,2026-10-01,23,36,767,False
10,YAT,Fon_Verileri_CSV_Tum_Veri_2026-10-10_nisan_men...,2026-04-01,2026-04-30,21,1989,41614,False
9,YAT,Fon_Verileri_CSV_Tum_Veri_2026-10-10_mayıs_men...,2026-05-04,2026-05-26,16,1996,31832,False
8,YAT,Fon_Verileri_CSV_Tum_Veri_2026-10-10_haziran_m...,2026-06-01,2026-06-30,22,2013,44093,False
11,YAT,Fon_Verileri_CSV_Tum_Veri_2026-10-10_temmuz_me...,2026-07-01,2026-07-31,22,2026,44413,False


### 1.2 Duplicates, invalid prices and the window

- A fund should have one price per day. If the same (fund, date) appears twice (for example overlapping exports), I keep the last one and report any disagreement.
- Prices of zero or below are not real prices; I drop those rows and count them.
- Rows outside the study window are removed.

In [17]:
dupes = tr_raw[tr_raw.duplicated(["symbol", "date"], keep=False)]
conflicts = dupes.groupby(["symbol", "date"])["price"].nunique().gt(1).sum() if len(dupes) else 0
tr = tr_raw.drop_duplicates(["symbol", "date"], keep="last")

bad_price = tr["price"].isna() | (tr["price"] <= 0)
outside = ~tr["date"].between(START, END)
print(f"Duplicate (fund, date) rows removed: {len(tr_raw) - len(tr)} (with different prices: {conflicts})")
print(f"Rows with missing/zero/negative price removed: {bad_price.sum()} "
      f"(funds affected: {tr.loc[bad_price, 'symbol'].nunique()})")
print(f"Rows outside the window removed: {outside.sum()}")

tr_bad_price_funds = set(tr.loc[bad_price, "symbol"])
tr = tr.loc[~bad_price & ~outside].sort_values(["symbol", "date"]).reset_index(drop=True)
print(f"Clean TR rows: {len(tr):,} | funds: {tr['symbol'].nunique():,}")

Duplicate (fund, date) rows removed: 0 (with different prices: 0)
Rows with missing/zero/negative price removed: 982 (funds affected: 41)
Rows outside the window removed: 0
Clean TR rows: 254,156 | funds: 2,085


### 1.3 Fund metadata

I derive a few labels from the fund name:
- **Private funds** (`ÖZEL FON`) are only open to the investors named in their internal rules, and **hedge funds** (`SERBEST FON`) only to qualified investors, so a normal investor cannot buy either. They will be excluded. (`ÖZEL SEKTÖR BORÇLANMA ARAÇLARI` means private-sector bonds; those are ordinary public bond funds and are kept.)
- A **category** (money market, bonds, Turkish equities, foreign equities, gold, participation/interest-free, ...). This is a description only; it is not used to pick or drop funds.
- **Fund size** on the first day of the window. Also description only (not a filter); it may be used as a fund characteristic in Phase 4.

In [18]:
def tr_category(name, fund_type):
    """Rough asset-class label from the TEFAS fund name (names are uppercase Turkish)."""
    if fund_type == "BYF":
        return "byf_" + tr_category(name, "YAT")  # same labels, marked as exchange-traded
    if "PARA PİYASASI" in name:
        return "money_market"
    if "ALTIN" in name or "GÜMÜŞ" in name or "KIYMETLİ MADEN" in name:
        return "gold"
    if "KATILIM" in name:
        return "participation"
    if "HİSSE" in name and "YABANCI" in name:
        return "equity_foreign"
    if "HİSSE" in name:
        return "equity_tr"
    if "BORÇLANMA" in name or "TAHVİL" in name or "BONO" in name:
        return "bond"
    if "DEĞİŞKEN" in name:
        return "flexible"
    if "FON SEPETİ" in name:
        return "fund_of_funds"
    if "KARMA" in name:
        return "mixed"
    return "other"


tr_meta = (tr.sort_values("date").groupby("symbol")
           .agg(name=("name", "last"), fund_type=("fund_type", "last"),
                first_date=("date", "min"), last_date=("date", "max"), n_obs=("date", "nunique"))
           .reset_index())
size_at_start = tr[tr["date"] == tr["date"].min()].set_index("symbol")["fund_size_tl"]
tr_meta["fund_size_tl_at_start"] = tr_meta["symbol"].map(size_at_start)
# "ÖZEL FON" = private fund; "ÖZEL SEKTÖR BORÇLANMA ARAÇLARI" = a normal public private-sector bond fund
tr_meta["is_private"] = tr_meta["name"].str.contains(r"ÖZEL(?! SEKTÖR)", regex=True)
tr_meta["is_hedge_fund"] = tr_meta["name"].str.contains("SERBEST", regex=False)
tr_meta["category"] = [tr_category(n, t) for n, t in zip(tr_meta["name"], tr_meta["fund_type"])]

display(pd.crosstab(tr_meta["category"], tr_meta["is_private"] | tr_meta["is_hedge_fund"],
                    margins=True).rename(columns={False: "public", True: "private/hedge"}))

col_0,public,private/hedge,All
category,,,
bond,77,9,86
byf_equity_tr,21,0,21
byf_gold,6,0,6
byf_other,4,0,4
byf_participation,5,0,5
equity_foreign,12,0,12
equity_tr,146,276,422
flexible,156,15,171
fund_of_funds,72,12,84


## 2. US ETFs

Created by `abd_etf_download.py` (Yahoo Finance via `yfinance`). Prices are in USD.

**Which price?** I use `adj_close` (adjusted close). It is the closing price corrected for dividends and splits, so its daily change is the **total return** (price change + dividends reinvested).
Because dividends are already inside `adj_close`, I must **not** add the `dividends` column on top (that would count them twice). I keep `close` and `dividends` only for documentation.

**Reproducibility note:** Yahoo recalculates `adj_close` backwards whenever a new dividend is paid, so re-downloading later can change old values slightly. The raw file with its `retrieved_at_utc` column is my frozen snapshot.

In [19]:
us_raw = pd.read_csv(US_FILE, encoding="utf-8-sig")
us_raw["date"] = pd.to_datetime(us_raw["date"], format="%Y-%m-%d")  # strict ISO format

us = us_raw.rename(columns={"ticker": "symbol", "fund_name": "name"}).copy()
us["price"] = us["adj_close"]
us["price_type"] = "yahoo_adj_close_total_return"

n_dup = us.duplicated(["symbol", "date"]).sum()
bad_price = us["price"].isna() | (us["price"] <= 0)
outside = ~us["date"].between(START, END)
us = us.drop_duplicates(["symbol", "date"], keep="last").loc[~bad_price & ~outside]
us = us.sort_values(["symbol", "date"]).reset_index(drop=True)
print(f"Duplicates: {n_dup} | invalid prices: {bad_price.sum()} | outside window: {outside.sum()}")
print(f"Clean US rows: {len(us):,} | ETFs: {us['symbol'].nunique():,} | "
      f"dates: {us['date'].min().date()} to {us['date'].max().date()}")
if us["date"].min() > START + pd.Timedelta(days=5) or us["date"].max() < END - pd.Timedelta(days=5):
    print("WARNING: the US file does not cover the whole window. Re-run abd_etf_indir.py with the same dates.")

# Tradability: share of the FIRST trading days on which the ETF actually traded (volume > 0).
# Only the first days are used, so no future information enters the rule.
first_days = sorted(us["date"].unique())[:US_TRADED_DAYS_WINDOW]
early = us[us["date"].isin(first_days)]
traded_share = (early["volume"].fillna(0) > 0).groupby(early["symbol"]).sum() / len(first_days)
# Median dollar volume is kept for documentation only (not a filter; volume is not allowed as a Phase 2 signal)
dollar_volume = (early["close"] * early["volume"]).groupby(early["symbol"]).median()

us_meta = (us.groupby("symbol")
           .agg(name=("name", "last"), first_date=("date", "min"), last_date=("date", "max"),
                n_obs=("date", "nunique"), n_dividends=("dividends", lambda s: int((s > 0).sum())))
           .reset_index())
us_meta["fund_type"] = "US_ETF"
us_meta["category"] = "us_etf"
us_meta["traded_share_start"] = us_meta["symbol"].map(traded_share).fillna(0)
us_meta["median_dollar_volume_start"] = us_meta["symbol"].map(dollar_volume)
us_meta["is_leveraged_or_inverse"] = us_meta["name"].fillna("").str.upper().str.contains(US_EXCLUDE_PATTERN, regex=True)
print(f"Leveraged / inverse ETFs (by name): {us_meta['is_leveraged_or_inverse'].sum():,}")
print("Examples (check these look right):")
display(us_meta.loc[us_meta["is_leveraged_or_inverse"], ["symbol", "name"]].sample(
    min(15, int(us_meta["is_leveraged_or_inverse"].sum())), random_state=0))

Duplicates: 0 | invalid prices: 0 | outside window: 0
Clean US rows: 676,639 | ETFs: 5,746 | dates: 2026-04-01 to 2026-10-01
Leveraged / inverse ETFs (by name): 819
Examples (check these look right):


,symbol,name
4632,SPOG,Leverage Shares 2X Long SPOT Daily ETF
4549,SMUP,T-REX 2X Long SMR Daily Target ETF
3808,PLTZ,Defiance Daily Target 2x Short PLTR ETF
5097,UECG,Leverage Shares 2x Long UEC Daily ETF
606,BRKU,Direxion Daily BRKB Bull 2X ETF
74,ADIU,Leverage Shares 2X Long ADI Daily ETF
167,AMDC,Corgi AMD 2x Daily ETF
3539,NVOX,Defiance Daily Target 2X Long NVO ETF
5220,UXI,ProShares Ultra Industrials
5054,TWM,ProShares UltraShort Russell2000


## 3. USD/TRY exchange rate

The portfolio mixes TRY and USD assets, so it needs one reporting currency and an exchange rate to convert between them.
`USDTRY` = how many Turkish lira one US dollar costs. If it goes from 40 to 41, a USD asset gains 2.5% in TRY terms even if its USD price did not move.

The rate is downloaded once and saved; later runs read the saved file, so results stay reproducible.

**Timing note (document in the report):** Yahoo's daily FX value is taken late in the day (London/New York time), while TEFAS prices are set during the Turkish day. They are close but not perfectly simultaneous.

In [20]:
fx = None
if FX_FILE.exists():
    fx = pd.read_csv(FX_FILE, parse_dates=["date"])
    print(f"Loaded saved FX file: {FX_FILE}")
else:
    try:
        import yfinance as yf
        raw_fx = yf.download(FX_TICKER, start=str(START.date()),
                             end=str((END + pd.Timedelta(days=1)).date()),
                             auto_adjust=False, progress=False, multi_level_index=False)
        if raw_fx.empty:
            raise ValueError("empty result")
        fx = (raw_fx[["Close"]].rename(columns={"Close": "usdtry"})
              .rename_axis("date").reset_index())
        fx["date"] = pd.to_datetime(fx["date"]).dt.tz_localize(None).dt.normalize()
        fx["source"] = f"Yahoo Finance {FX_TICKER} (daily close)"
        fx["retrieved_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
        fx.to_csv(FX_FILE, index=False)
        print(f"Downloaded and saved: {FX_FILE}")
    except Exception as error:
        print(f"WARNING: could not get USD/TRY ({error}).")
        print("If the error is \"No module named 'yfinance'\", this notebook's kernel is a different Python")
        print("environment from the one that ran abd_etf_indir.py. Fix: run  %pip install yfinance  in a cell,")
        print("restart the kernel, and run all cells again (or switch the kernel to the project's .venv).")

if fx is not None:
    fx = fx[fx["date"].between(START, END) & (fx["usdtry"] > 0)].sort_values("date").reset_index(drop=True)
    print(f"FX rows: {len(fx)} | {fx['date'].min().date()} to {fx['date'].max().date()} | "
          f"USDTRY {fx['usdtry'].iloc[0]:.4f} -> {fx['usdtry'].iloc[-1]:.4f}")

Loaded saved FX file: Data\Raw\fx\usdtry_daily_2026-04-01_2026-10-01.csv
FX rows: 132 | 2026-04-01 to 2026-10-01 | USDTRY 44.4549 -> 49.0280


## 4. Data quality checks

### 4.1 Trading calendars
Turkey and the US have different holidays. I list the weekdays with no prices in each market, so I can confirm they are real holidays and not missing data.

In [21]:
weekdays = pd.bdate_range(START, END)
tr_days = pd.DatetimeIndex(sorted(tr["date"].unique()))
us_days = pd.DatetimeIndex(sorted(us["date"].unique()))

for label, days in [("TR", tr_days), ("US", us_days)]:
    missing = weekdays.difference(days)
    print(f"{label}: {len(days)} trading days | weekdays without prices ({len(missing)}): "
          f"{', '.join(d.strftime('%d %b') for d in missing)}")

print(f"\nTR-only days: {len(tr_days.difference(us_days))} | US-only days: {len(us_days.difference(tr_days))} | "
      f"common days: {len(tr_days.intersection(us_days))}")

TR: 125 trading days | weekdays without prices (7): 23 Apr, 01 May, 19 May, 27 May, 28 May, 29 May, 15 Jul
US: 127 trading days | weekdays without prices (5): 03 Apr, 25 May, 19 Jun, 03 Jul, 07 Sep

TR-only days: 5 | US-only days: 7 | common days: 120


### 4.2 Suspicious jumps
Daily returns are computed for each asset **on its own trading days** (no filling of gaps). A fund that moves more than the limit in one day usually has a data issue (a unit-price reset, a split, a typo) rather than a real market move.

In [22]:
def max_abs_daily_return(df):
    returns = df.sort_values(["symbol", "date"]).groupby("symbol")["price"].pct_change()
    return returns.abs().groupby(df.sort_values(["symbol", "date"])["symbol"]).max()


tr_meta["max_abs_daily_return"] = tr_meta["symbol"].map(max_abs_daily_return(tr))
us_meta["max_abs_daily_return"] = us_meta["symbol"].map(max_abs_daily_return(us))
tr_meta["suspicious_jump"] = tr_meta["max_abs_daily_return"] > MAX_DAILY_MOVE_TR
us_meta["suspicious_jump"] = us_meta["max_abs_daily_return"] > MAX_DAILY_MOVE_US
tr_meta["had_invalid_price"] = tr_meta["symbol"].isin(tr_bad_price_funds)

print(f"TR funds with a jump > {MAX_DAILY_MOVE_TR:.0%}: {tr_meta['suspicious_jump'].sum()}")
print(f"US ETFs  with a jump > {MAX_DAILY_MOVE_US:.0%}: {us_meta['suspicious_jump'].sum()}")
display(tr_meta.loc[tr_meta["suspicious_jump"] & ~tr_meta["is_private"] & ~tr_meta["is_hedge_fund"],
                    ["symbol", "name", "category", "max_abs_daily_return"]].head(10))

TR funds with a jump > 25%: 63
US ETFs  with a jump > 40%: 137


,symbol,name,category,max_abs_daily_return
1410,PBR,PUSULA PORTFÖY BİRİNCİ DEĞİŞKEN FON,flexible,0.599736
1433,PHE,PUSULA PORTFÖY HİSSE SENEDİ FONU (HİSSE SENEDİ...,equity_tr,0.478093
1524,PUK,PUSULA PORTFÖY KATILIM HİSSE SENEDİ (TL) FONU ...,participation,0.291170
1732,THF,TERA PORTFÖY HİSSE SENEDİ (TL) FONU (HİSSE SEN...,equity_tr,0.386448


## 5. Eligibility filters

Each filter removes assets that **cannot be used**, for one documented reason. Nothing is removed for being small, unpopular or performing badly. The table below (a "funnel") shows how many assets survive each step and why each step exists; it can go straight into the report.

**Coverage** = share of the market's trading days on which the asset has a price. An asset that was launched in the middle of the window, or is missing from an export, fails this rule because it cannot be simulated or validated over the window.

In [23]:
tr_meta["coverage"] = tr_meta["n_obs"] / len(tr_days)
us_meta["coverage"] = us_meta["n_obs"] / len(us_days)

# (label, reason, rule) - rule is True for assets that are KEPT
tr_rules = [
    ("all funds in the exports", "", pd.Series(True, index=tr_meta.index)),
    ("not a private fund (ÖZEL)", "access: only for specific investors, a normal investor cannot buy it",
     ~tr_meta["is_private"]),
    ("not a hedge fund (SERBEST)", "access: only for qualified investors",
     ~tr_meta["is_hedge_fund"]),
    (f"coverage >= {MIN_COVERAGE:.0%}", "data: not enough prices to simulate over the window",
     tr_meta["coverage"] >= MIN_COVERAGE),
    ("no zero/invalid prices", "data: a zero or missing price is not a real price",
     ~tr_meta["had_invalid_price"]),
    (f"no daily jump > {MAX_DAILY_MOVE_TR:.0%}", "data: such a jump is almost always a price error or reset",
     ~tr_meta["suspicious_jump"]),
]
us_rules = [
    ("all ETFs in the download", "", pd.Series(True, index=us_meta.index)),
    ("not leveraged / inverse", "project rule: no short selling or borrowing",
     ~us_meta["is_leveraged_or_inverse"]),
    (f"coverage >= {MIN_COVERAGE:.0%}", "data: not enough prices to simulate over the window",
     us_meta["coverage"] >= MIN_COVERAGE),
    (f"no daily jump > {MAX_DAILY_MOVE_US:.0%}", "data: such a jump is almost always a price error or split",
     ~us_meta["suspicious_jump"]),
    (f"traded on all of the first {US_TRADED_DAYS_WINDOW} days", "tradability: with zero volume the close is not an executable price",
     us_meta["traded_share_start"] >= MIN_US_TRADED_SHARE),
]


def apply_rules(meta, rules, market):
    keep = pd.Series(True, index=meta.index)
    meta["excluded_by"] = ""
    rows = []
    for label, reason, rule in rules:
        newly_out = keep & ~rule.fillna(False)
        meta.loc[newly_out, "excluded_by"] = label
        keep &= rule.fillna(False)
        rows.append({"market": market, "step": label, "reason": reason,
                     "removed": int(newly_out.sum()), "remaining": int(keep.sum())})
    meta["eligible"] = keep
    return pd.DataFrame(rows)


funnel = pd.concat([apply_rules(tr_meta, tr_rules, "TR"), apply_rules(us_meta, us_rules, "US")], ignore_index=True)
display(funnel)
display(tr_meta[tr_meta["eligible"]].groupby(["fund_type", "category"]).size().rename("eligible_TR_funds").to_frame())

,market,step,reason,removed,remaining
0,TR,all funds in the exports,,0,2085
1,TR,not a private fund (ÖZEL),"access: only for specific investors, a normal ...",484,1601
2,TR,not a hedge fund (SERBEST),access: only for qualified investors,827,774
3,TR,coverage >= 98%,data: not enough prices to simulate over the w...,67,707
4,TR,no zero/invalid prices,data: a zero or missing price is not a real price,0,707
5,TR,no daily jump > 25%,data: such a jump is almost always a price err...,0,707
6,US,all ETFs in the download,,0,5746
7,US,not leveraged / inverse,project rule: no short selling or borrowing,819,4927
8,US,coverage >= 98%,data: not enough prices to simulate over the w...,556,4371
9,US,no daily jump > 40%,data: such a jump is almost always a price err...,3,4368


eligible_TR_funds
fund_type category                            
BYF       byf_equity_tr                     17
          byf_gold                           5
          byf_other                          4
          byf_participation                  4
YAT       bond                              77
          equity_foreign                    12
          equity_tr                        136
          flexible                         141
          fund_of_funds                     69
          gold                              51
          mixed                              8
          money_market                      97
          other                              4
          participation                     82

## 6. Eligible universe

**No hand-picking.** The universe is simply every asset that passed the eligibility filters in step 5, in both markets. I do not choose "the best" or "the biggest" funds here, because choosing is the job of the Phase 2 algorithm (using price-based signals only). The Phase 3 benchmarks (equal weight, buy and hold) use this same eligible universe.

If a smaller universe is ever needed (for speed, for example), it must come from a rule written here and applied the same way to every asset, never from a list typed by hand.

In [24]:
tr_universe = tr_meta[tr_meta["eligible"]]
us_universe = us_meta[us_meta["eligible"]]

print(f"Eligible universe: {len(tr_universe)} TR funds + {len(us_universe)} US ETFs "
      f"= {len(tr_universe) + len(us_universe)} assets")
display(tr_universe.groupby(["fund_type", "category"]).size().rename("funds").to_frame())

Eligible universe: 707 TR funds + 4201 US ETFs = 4908 assets


funds
fund_type category                
BYF       byf_equity_tr         17
          byf_gold               5
          byf_other              4
          byf_participation      4
YAT       bond                  77
          equity_foreign        12
          equity_tr            136
          flexible             141
          fund_of_funds         69
          gold                  51
          mixed                  8
          money_market          97
          other                  4
          participation         82

## 7. Save outputs

I add trading and settlement assumptions for each instrument type (Phase 1 asks for these so the later simulation can be realistic). They are **assumptions to verify**, especially for TEFAS funds, where each fund's rules are published on its TEFAS/KAP page.

| Type | Where it trades | Price I use | Settlement assumption |
|---|---|---|---|
| US ETF | NYSE / Nasdaq, during the day | adjusted close | T+1 |
| BYF | Borsa İstanbul, during the day | bulletin (closing) price | T+2 |
| TEFAS fund | TEFAS order, **forward pricing**: the order is filled at a price that is not known when you submit it | unit price | varies by fund (often T+0 to T+3) |

**Files written to `Data/Processed/`**

| File | Content |
|---|---|
| `prices_long.csv` | one row per (date, asset) for all **eligible** assets, local currency, no gap filling |
| `instruments.csv` | every asset seen, with metadata, quality flags, eligibility and the reason for exclusion (data dictionary) |
| `universe.csv` | the eligible universe (list of asset IDs) |
| `universe_prices_wide.csv` | eligible universe as a table: dates x assets, local currency, **gaps not filled** |
| `fx_usdtry.csv` | USD/TRY daily rate (if available) |
| `data_quality_report.csv` | per-file coverage + eligibility funnel with the reason for each step |

In [25]:
TRADING_RULES = {
    "US_ETF": ("NYSE/Nasdaq exchange", "USD", "T+1"),
    "BYF": ("Borsa Istanbul exchange", "TRY", "T+2"),
    "YAT": ("TEFAS order, forward pricing", "TRY", "varies by fund (often T+0 to T+3), verify on TEFAS/KAP"),
}

tr_meta["market"], us_meta["market"] = "TR", "US"
instruments = pd.concat([tr_meta, us_meta], ignore_index=True)
instruments["asset_id"] = instruments["market"] + ":" + instruments["symbol"]
instruments["trading_venue"] = instruments["fund_type"].map(lambda t: TRADING_RULES[t][0])
instruments["currency"] = instruments["fund_type"].map(lambda t: TRADING_RULES[t][1])
instruments["settlement_assumption"] = instruments["fund_type"].map(lambda t: TRADING_RULES[t][2])
universe_ids = set(instruments.loc[instruments["eligible"], "asset_id"])  # universe = all eligible assets
instruments["in_universe"] = instruments["asset_id"].isin(universe_ids)

first_cols = ["asset_id", "market", "symbol", "name", "fund_type", "category", "currency", "eligible",
              "excluded_by", "in_universe", "first_date", "last_date", "n_obs", "coverage",
              "max_abs_daily_return", "trading_venue", "settlement_assumption"]
instruments = instruments[first_cols + [c for c in instruments.columns if c not in first_cols]]

keep_cols = ["date", "asset_id", "market", "symbol", "currency", "price", "price_type"]
tr["market"], tr["currency"] = "TR", "TRY"
us["market"], us["currency"] = "US", "USD"
prices_long = pd.concat([tr, us], ignore_index=True)
prices_long["asset_id"] = prices_long["market"] + ":" + prices_long["symbol"]
eligible_ids = set(instruments.loc[instruments["eligible"], "asset_id"])
prices_long = (prices_long.loc[prices_long["asset_id"].isin(eligible_ids), keep_cols]
               .sort_values(["date", "asset_id"]).reset_index(drop=True))

universe_wide = (prices_long[prices_long["asset_id"].isin(universe_ids)]
                 .pivot(index="date", columns="asset_id", values="price").sort_index())

quality_report = pd.concat([
    coverage_by_file.assign(section="tr_file_coverage"),
    funnel.assign(section="eligibility_funnel"),
], ignore_index=True)

prices_long.to_csv(OUT / "prices_long.csv", index=False)
instruments.to_csv(OUT / "instruments.csv", index=False, encoding="utf-8-sig")
instruments.loc[instruments["in_universe"], ["asset_id", "market", "symbol", "name", "category", "currency"]] \
    .to_csv(OUT / "universe.csv", index=False, encoding="utf-8-sig")
universe_wide.to_csv(OUT / "universe_prices_wide.csv")
quality_report.to_csv(OUT / "data_quality_report.csv", index=False, encoding="utf-8-sig")
if fx is not None:
    fx.to_csv(OUT / "fx_usdtry.csv", index=False)

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")

data_quality_report.csv        0.00 MB
fx_usdtry.csv                  0.01 MB
instruments.csv                1.81 MB
prices_long.csv               46.99 MB
universe.csv                   0.34 MB
universe_prices_wide.csv      10.51 MB


## 8. Final validation checks

These checks fail loudly (`AssertionError`) if something is wrong, so a bad dataset cannot slip into Phase 2 unnoticed.

In [26]:
checks = {
    "1. no duplicate (date, asset) rows": not prices_long.duplicated(["date", "asset_id"]).any(),
    "2. all prices positive": (prices_long["price"] > 0).all(),
    "3. all dates inside the window": prices_long["date"].between(START, END).all(),
    "4. both markets in the eligible universe": len(tr_universe) > 0 and len(us_universe) > 0,
    "5. universe table has no all-empty column": universe_wide.notna().any().all(),
    "6. dates sorted and unique in universe table": universe_wide.index.is_monotonic_increasing and universe_wide.index.is_unique,
}
for number, label, days in [(7, "TR", tr_days), (8, "US", us_days)]:
    spans = days.min() <= START + pd.Timedelta(days=5) and days.max() >= END - pd.Timedelta(days=5)
    checks[f"{number}. {label} data spans the whole window ({days.min().date()} to {days.max().date()})"] = spans
if fx is None:
    checks["9. USD/TRY exchange rate loaded (needed for a combined portfolio)"] = False
else:
    missing_fx = us_days.union(tr_days).difference(pd.DatetimeIndex(fx["date"]))
    checks["9. FX available on (almost) every trading day"] = len(missing_fx) <= 10

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some checks failed: see above."

print(f"\nEligible universe: {universe_wide.shape[1]} assets x {universe_wide.shape[0]} dates "
      f"({universe_wide.index.min().date()} to {universe_wide.index.max().date()})")

PASS  1. no duplicate (date, asset) rows
PASS  2. all prices positive
PASS  3. all dates inside the window
PASS  4. both markets in the eligible universe
PASS  5. universe table has no all-empty column
PASS  6. dates sorted and unique in universe table
PASS  7. TR data spans the whole window (2026-04-01 to 2026-10-01)
PASS  8. US data spans the whole window (2026-04-01 to 2026-10-01)
PASS  9. FX available on (almost) every trading day

Eligible universe: 4908 assets x 132 dates (2026-04-01 to 2026-10-01)


## How to use the outputs in Phase 2

```python
prices = pd.read_csv("Data/Processed/universe_prices_wide.csv", index_col="date", parse_dates=True)
fx = pd.read_csv("Data/Processed/fx_usdtry.csv", index_col="date", parse_dates=True)["usdtry"]
```

Gaps are **not** filled on purpose: a Turkish holiday is a day where TR funds have no price, and filling it would create fake 0% returns that distort volatility and correlation. Decide how to align the two calendars inside the algorithm (for weekly decisions, taking the last available price of each week is the simplest approach: `prices.resample("W-FRI").last()`).

The algorithm, not this notebook, decides which eligible assets to hold.

Every column of every output file is described in `DATA_DICTIONARY.md`.



### Ezgi Dönmez
